In [2]:
import requests
import psycopg2
import time

In [4]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="megabase0",
    user="postgres",
    password="toto"
)


cur = conn.cursor()

cur.execute("""
    SELECT insee_code, name
    FROM commune
    LIMIT 10;
    
""")

rows = cur.fetchall()

code_insee=[]

for row in rows:
    code_insee.append(row[0])

#print(code_insee)

TOKEN="c624deda9bddf361b3e5f11b4af95e16b324808b49396ee39f145e8a0274d5e1"
url = 'https://api.meteo-concept.com/api/ephemeride/1'

for insee in code_insee:

    params = {
        "token": TOKEN,
        "insee": insee
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()

        ephemeride = data["ephemeride"]        

        insee_code=ephemeride["insee"]
        ephemeride_date=ephemeride["datetime"]
        sunrise = ephemeride["sunrise"]
        sunset = ephemeride["sunset"]
        day_duration = ephemeride["duration_day"]

        cur.execute("""
            INSERT INTO ephemeride(
                insee_code,
                ephemeride_date,
                sunrise,
                sunset,
                day_duration
            )
            VALUES (%s, %s, %s, %s, %s)
        """,
        (
            insee_code,
            ephemeride_date,
            sunrise,
            sunset,
            day_duration
        ))

        print("Inséré :", insee)

    else:
        print("Requête ne passe pas :", insee, response.status_code, response.text)

    time.sleep(0.2)

conn.commit()

cur.close()
conn.close()


Inséré : 69001
Inséré : 69002
Inséré : 69003
Inséré : 69004
Inséré : 69005
Inséré : 69006
Inséré : 69007
Inséré : 69008
Inséré : 69009
Inséré : 69010


In [6]:
import requests

TOKEN = "c624deda9bddf361b3e5f11b4af95e16b324808b49396ee39f145e8a0274d5e1"

url = "https://api.meteo-concept.com/api/ephemeride/0"

params = {
    "token": TOKEN,
    "insee": "35238"
}

response = requests.get(url, params=params)

# print(response.url)
# print(response.status_code)
# print(response.text)
data=response.json()
from pprint import pprint
pprint(data['ephemeride'])

{'datetime': '2026-06-30T00:00:00+0200',
 'day': 0,
 'diff_duration_day': 0,
 'duration_day': '16:04',
 'insee': '35238',
 'latitude': 48.112,
 'longitude': -1.6819,
 'moon_age': 14.7,
 'moon_phase': 'Pleine lune',
 'sunrise': '06:08:00',
 'sunset': '22:12:00'}
